#Find Consecutive Customer Activity Days

## Business Scenario

An e-commerce company wants to identify customers **who logged into the platform for **3 or more consecutive days****.

The marketing team believes that customers who are active continuously for several days are highly engaged and more likely to make purchases.

## Business Questions

- Which customers are highly engaged?
- Who should receive loyalty rewards?
- Which users are likely to convert into premium customers?
- Which customers show a buying intent pattern?

## Sample Data

**`customer_activity`**

| customer_id | activity_date |
| ----------- | ------------- |
| 101         | 2026-11-01    |
| 101         | 2026-11-02    |
| 101         | 2026-11-03    |
| 101         | 2026-11-06    |
| 102         | 2026-11-01    |
| 102         | 2026-11-03    |
| 102         | 2026-11-04    |
| 103         | 2026-11-05    |
| 103         | 2026-11-06    |
| 103         | 2026-11-07    |
| 103         | 2026-11-08    |

## Expected Output

| customer_id | consecutive_days |
| ----------- | ---------------- |
| 101         | 3                |
| 103         | 4                |

**Notes:**

- Customer 101 was active for 3 consecutive days.
- Customer 103 was active for 4 consecutive days.

|customer_id|activity_date|rn|grp|
|---|---|---|---|
|101|2026-11-01|1|2026-10-31|
|101|2026-11-02|2|2026-10-31|
|101|2026-11-03|3|2026-10-31|
|101|2026-11-06|4|2026-11-02|
|102|2026-11-01|1|2026-10-31|
|102|2026-11-03|2|2026-11-01|
|102|2026-11-04|3|2026-11-01|
|103|2026-11-05|1|2026-11-04|
|103|2026-11-06|2|2026-11-04|
|103|2026-11-07|3|2026-11-04|
|103|2026-11-08|4|2026-11-04|

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

data = [
(101,'2026-11-01'),
(101,'2026-11-02'),
(101,'2026-11-03'),
(101,'2026-11-06'),

(102,'2026-11-01'),
(102,'2026-11-03'),
(102,'2026-11-04'),

(103,'2026-11-05'),
(103,'2026-11-06'),
(103,'2026-11-07'),
(103,'2026-11-08')
]

df = spark.createDataFrame(
    data,
    ["customer_id","activity_date"]
).withColumn(
    "activity_date",
    F.to_date("activity_date")
)

In [0]:
df.display()

In [0]:
window_spec = Window.partitionBy("customer_id") \
                    .orderBy("activity_date")

In [0]:
df1 = df.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df1.display()

In [0]:
df2 = df1.withColumn(
    "grp",
    F.date_sub(
        F.col("activity_date"),
        F.col("rn")
    )
)
df2.display()

In [0]:
result = (
    df2.groupBy("customer_id","grp")
       .agg(
           F.min("activity_date").alias("start_date"),
           F.max("activity_date").alias("end_date"),
           F.count("*").alias("consecutive_days")
       )
       
)
result.show()

In [0]:

result.filter("consecutive_days >= 3").display()
